# Lab 2 · Find the weird transactions — Unsupervised Learning

**Fundamentals of AI** · Same data as Lab 1: [Credit Card Fraud Detection (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

New story: **nobody has labeled anything.** A brand-new fraud pattern appeared — can the machine still find it?

# 🗺️ The ML project pipeline — where you are

Same skeleton as any ML project — the twist is **no labels** (that's what *unsupervised* means):

| # | Pipeline step | Where in this lab |
|---|---------------|-------------------|
| 1 | **Get the data** | §1 Setup & load |
| 2 | **Clean & pre-process** — audit, drop duplicates, scale features | §2 Prepare the data |
| 3 | **Choose a technique** — cluster *or* find outliers | §3 K-Means · §5 Isolation Forest |
| 4 | **Look at it** — visualize in 2D | §4 PCA |
| 5 | **Evaluate** — score the flags against hidden truth | §6 The reveal |
| 6 | **Tune the decision** — pick flag_pct by cost | §7 Cost game |
| 7 | **Export & wrap up** — save models, what you learned | §8 Export · §9 Takeaways |

## 1 · Get the data — Setup & load

**Install libraries** (safe to skip on Colab).

In [ ]:
#@title 0. Install libraries { display-mode: "form" }
# Colab already has these; this makes the notebook work anywhere. -q = quiet.
%pip install -q --disable-pip-version-check scikit-learn pandas matplotlib ipywidgets


**Imports + helpers** — theme colors, `scatter2()` for 2D plots, `need()` guard.

In [ ]:
#@title 1a. Theme + helpers { display-mode: "form" }
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# dark theme colors used by all charts (RED = flagged anomalies)
BG, ACCENT, ACCENT2, TEXT, RED = "#030503", "#00E676", "#3DFF8A", "#FFFFFF", "#FF5252"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "axes.titlecolor": TEXT,
    "grid.color": "#1c3a24", "font.size": 14, "axes.titlesize": 18,
    "axes.labelsize": 15, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "legend.facecolor": "#0a140d", "legend.edgecolor": ACCENT,
    "figure.figsize": (8, 5),
})

def scatter2(Z, c=None, highlight=None, title=""):
    """2D scatter on the dark theme. highlight = boolean mask drawn in red."""
    fig, ax = plt.subplots()
    # main points (optionally colored by cluster), small & semi-transparent
    ax.scatter(Z[:, 0], Z[:, 1], c=c if c is not None else ACCENT2,
               s=4, alpha=0.5, linewidths=0)
    # overlay flagged points in red on top
    if highlight is not None and highlight.any():
        ax.scatter(Z[highlight, 0], Z[highlight, 1], c=RED, s=10, alpha=0.9,
                   linewidths=0, label="flagged")
        ax.legend()
    ax.set_xlabel("component 1"); ax.set_ylabel("component 2")
    ax.set_title(title)
    plt.tight_layout(); plt.show()

def need(*names):
    """Friendly stop if an earlier step hasn't run yet."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise SystemExit(f"✏️ Not ready yet — missing: {', '.join(missing)}. Run the earlier cells first.")
print("Theme ready.")


**Load the data** — local CSV first, OpenML fallback; labels are hidden in `answer_key` until §6.

In [ ]:
#@title 1b. Load the data { display-mode: "form" }
import os

# look for a local CSV first (data/creditcard.csv, parent dir, or notebook dir)
csv_path = next((p for p in ["data/creditcard.csv", "../data/creditcard.csv", "creditcard.csv"]
                 if os.path.exists(p)), None)

if csv_path:
    print(f"Loading local file: {csv_path}")
    _df = pd.read_csv(csv_path)
    X_all = _df.drop(columns=["Class"])      # features only
    _y = _df["Class"].astype(int).values     # real labels — kept aside for now
else:
    try:
        from sklearn.datasets import fetch_openml
        # OpenML mirror of the Kaggle dataset — no login needed
        data = fetch_openml(data_id=1597, as_frame=True, parser="auto")
        X_all = data.data if hasattr(data, "data") else data[0]
        _y = (data.target if hasattr(data, "target") else data[1]).astype(int).values
    except Exception as e:
        print("OpenML download failed:", e)
        print("FALLBACK: download creditcard.csv from")
        print("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  (free login),")
        print("put it in a 'data/' folder next to this notebook (or upload via the file panel),")
        print("then re-run this cell.")
        raise SystemExit("Dataset not loaded yet.")
print("Loaded OK.")

# 🔒 THE ANSWER KEY — locked until section 6. Do not peek!
answer_key = _y.copy()   # store real labels under a different name
y_all = None             # the models never see these
print(f"Loaded {X_all.shape[0]:,} transactions. Labels locked in 'answer_key'.")

**Peek at the data** — note there's no `Class` column; labels are locked away.

In [ ]:
#@title 1c. Peek at the data { display-mode: "form" }
print(f"Shape: {X_all.shape[0]:,} rows × {X_all.shape[1]} columns")
X_all.head()   # note: no 'Class' column — labels are locked away

> If 1b fails: download `creditcard.csv` from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), upload it, re-run 1b.

## 2 · Prepare the data — scale the features

Strip the labels: the machine only sees `X_all`. Different features live on wildly
different scales (`Amount` ~ hundreds, `V1`–`V28` ~ single digits) — so we scale them.

**Audit & clean first** — real data is messy. Check for missing values, duplicates and odd amounts *before* any modeling.

In [ ]:
#@title 2a. Audit the raw data { display-mode: "form" }
need("X_all")

print(f"Rows: {len(X_all):,}   Columns: {X_all.shape[1]}")
missing = X_all.isna().sum()
print("Missing values:", int(missing.sum()), "(none expected — PCA output is clean)")
print(f"Exact duplicate rows: {int(X_all.duplicated().sum()):,}")
print(f"Amount == 0: {int((X_all['Amount'] == 0).sum()):,}   "
      f"Amount < 0: {int((X_all['Amount'] < 0).sum()):,}")
print(f"Amount range: {X_all['Amount'].min():.2f} → {X_all['Amount'].max():.2f}")

In [ ]:
#@title 2b. Clean the data { display-mode: "form" }
need("X_all")

# drop exact duplicate rows — same transaction twice adds no information
keep = ~X_all.duplicated()                  # True = first occurrence of each row
n_dropped = int((~keep).sum())
X_all = X_all[keep].copy()                  # features, duplicates removed
# the hidden labels must drop the SAME rows or §6 scores would be wrong
if "answer_key" in globals():
    answer_key = answer_key[keep.to_numpy()]
if "_y" in globals():
    _y = _y[keep.to_numpy()]
print(f"Dropped {n_dropped:,} duplicate rows → {len(X_all):,} rows left")

**Scale the features** — every column to mean 0, std 1 so they count equally.

In [ ]:
# Scale features so every column counts equally (each → mean 0, std 1)
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X_all)
print("Scaled: each feature now has mean ≈ 0, std ≈ 1")

<details><summary>💡 Checkpoint — why scale before K-Means?</summary>

K-Means measures **distance** between points. If `Amount` ranges 0–25,000 while `V1`–`V28`
range −10–10, distances are dominated by `Amount` alone — the PCA features barely matter.
Scaling puts everything on equal footing so *all* features vote.</details>

## 3 · Choose a technique — K-Means clustering

K-Means groups similar transactions into `k` clusters — no labels needed.
Move the sliders and re-run.

**Run K-Means** — move the sliders for k and scaled/raw features.

In [ ]:
#@title 3a. K-Means — knobs
#@param {"type":"slider","min":2,"max":8,"step":1,"label":"k (number of clusters)"}
k = 4
#@param {"type":"boolean","label":"use scaled features? (uncheck to see Amount dominate)"}
use_scaling = True

need("X_all")
from sklearn.cluster import KMeans   # groups similar points into k clusters

# choose which feature set to cluster on
if use_scaling:
    need("X_scaled")          # needs Blank #1
    X_feat = X_scaled
else:
    X_feat = X_all.values     # raw features — Amount will dominate distances

# random sample of up to 50k rows — K-Means on 284k rows would be slow
rng = np.random.RandomState(42)   # fixed seed → same sample every run
idx = rng.choice(len(X_feat), size=min(50_000, len(X_feat)), replace=False)
X_s = X_feat[idx]                 # the sample we actually cluster & plot

kmeans = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_s)
labels_s = kmeans.labels_         # cluster id (0..k-1) for each sampled row
print(f"k={k}, scaling={'ON' if use_scaling else 'OFF'} — fitted on {len(X_s):,} transactions")
print("Cluster sizes (sample):", np.bincount(labels_s))

**Elbow plot** — try k from 2 to 10 and watch inertia drop.

In [ ]:
#@title 3b. Elbow plot — pick k { display-mode: "form" }
need("X_s")
from sklearn.cluster import KMeans

# for each k, fit K-Means and record inertia (total within-cluster distance)
ks = range(2, 11)
inertias = [KMeans(n_clusters=kk, n_init=10, random_state=0).fit(X_s[:20000]).inertia_
            for kk in ks]   # 20k subsample keeps this fast

fig, ax = plt.subplots()
ax.plot(ks, inertias, "o-", color=ACCENT, markersize=8)
ax.set_xlabel("k (clusters)"); ax.set_ylabel("inertia (within-cluster spread)")
ax.set_title("Elbow plot — where does adding clusters stop helping?")
ax.grid(True)
plt.tight_layout(); plt.show()

**Cluster sizes** — how many transactions landed in each cluster.

In [ ]:
#@title 3c. Cluster sizes { display-mode: "form" }
need("labels_s", "k")
fig, ax = plt.subplots()
sizes = np.bincount(labels_s, minlength=k)   # count rows per cluster id
ax.bar(range(k), sizes, color=ACCENT2, width=0.6)
ax.set_xlabel("cluster"); ax.set_ylabel("transactions (sample)")
ax.set_title("How many transactions landed in each cluster")
for i, v in enumerate(sizes):               # print count above each bar
    ax.text(i, v*1.02, f"{v:,}", ha="center", fontsize=13)
plt.tight_layout(); plt.show()

<details><summary>💡 Checkpoint — read the elbow</summary>

**Inertia** = total distance of points to their cluster center. More clusters → lower
inertia, always. The "elbow" is where gains flatten — past that point you're just
splitting noise. There's no single right k; we picked one to see the mechanics.</details>

## 4 · Look at the data — PCA to 2D

30 dimensions is impossible to plot. **PCA** squeezes them into 2 components that keep
as much information as possible — then we can *see* the clusters.
Colored by cluster — **not** by fraud (we don't have labels yet!).

**Project to 2D with PCA** and color by cluster.

In [ ]:
#@title 4. PCA → 2D scatter { display-mode: "form" }
need("X_s", "labels_s")
from sklearn.decomposition import PCA   # squeezes 30 dims → 2 components

pca = PCA(n_components=2, random_state=0).fit(X_s)
Z_s = pca.transform(X_s)                 # project sample into 2D
kept = pca.explained_variance_ratio_.sum() * 100   # % of info retained
print(f"2 components keep {kept:.1f}% of the information in all 30 features")

# color points by cluster id — NOT by fraud (labels still locked)
scatter2(Z_s, c=labels_s, title="Transactions in 2D (PCA), colored by K-Means cluster")

<details><summary>💡 What is PCA doing, in one line?</summary>

It finds the two directions along which the data spreads out the most, then projects
everything onto them. We lose ~some detail (the other 28 dims) but gain a picture.</details>

## 5 · Another technique — Isolation Forest

Different idea: instead of grouping, find the **weird** points. Isolation Forest isolates
random points with random splits — anomalies need fewer splits to be isolated ("easy to
isolate"). Move the slider to flag the top X% most unusual.

**Flag the most unusual** — Isolation Forest scores every transaction; the bottom X% get flagged.

In [ ]:
#@title 5a. Flag the most unusual
#@param {"type":"slider","min":0.5,"max":5,"step":0.5,"label":"flag top X% most unusual"}
flag_pct = 5   # slider value: flag the bottom X% most unusual

need("X_s", "X_feat")
from sklearn.ensemble import IsolationForest   # finds outliers via random splits

iforest = IsolationForest(n_estimators=100, random_state=0).fit(X_s)
# score every row (not just sample); LOWER score = more unusual
scores = iforest.score_samples(X_feat)
cutoff = np.quantile(scores, flag_pct / 100)    # threshold: bottom X% are "unusual"
flagged = scores <= cutoff                      # boolean mask over ALL rows
flag_s = flagged[idx]                           # same flags, restricted to plotted sample

print(f"Flagged {flagged.sum():,} transactions ({flag_pct}% of {len(X_feat):,})")

**See the flags on the map** — red = flagged as unusual.

In [ ]:
#@title 5b. Flags on the PCA map { display-mode: "form" }
need("Z_s", "flag_s", "labels_s", "flag_pct")
# same 2D map as before, now with flagged points overlaid in red
scatter2(Z_s, c=labels_s, highlight=flag_s,
         title=f"Red = flagged as unusual (top {flag_pct}%)")

<details><summary>💡 Checkpoint — what does Isolation Forest assume?</summary>

That anomalies are **few and different**: they sit alone, so random splits isolate them
quickly. It knows nothing about fraud — it just finds statistical oddballs. Whether an
oddball is fraud is a question for… the reveal.</details>

## 6 · Evaluate — the reveal

🔓 **Unlocking `answer_key`.** Now we check: of the transactions the machine flagged,
how many were *actually* fraud?

**Count the flagged frauds.**

In [ ]:
# Count how many flagged transactions are truly fraud
need("flagged", "answer_key")
n_flagged = int(flagged.sum())        # total rows the machine flagged
total_fraud = int(answer_key.sum())   # all real frauds in the dataset
# answer_key[flagged] keeps only the flagged rows → sum = truly flagged frauds
fraud_caught = int(answer_key[flagged].sum())
print(f"Flagged: {n_flagged:,}   Truly fraud among them: {fraud_caught:,}   (of {total_fraud:,} total frauds)")

**Scorecard** — precision, recall, and lift for the flags.

In [ ]:
#@title 6b. Scorecard — precision, recall, lift { display-mode: "form" }
need("fraud_caught", "n_flagged", "total_fraud")
N = len(answer_key)                    # all transactions
precision = fraud_caught / n_flagged   # of flags, share that are real fraud
recall    = fraud_caught / total_fraud # of all frauds, share we caught
base_rate = total_fraud / N            # fraud rate in the whole dataset
lift      = precision / base_rate      # how much richer flags are vs average

print(f"Flagged {n_flagged:,} transactions")
print(f"  truly fraud:        {fraud_caught:,}")
print(f"  precision:          {precision:.3f}   (1 in {1/precision:.0f} flags is real fraud)")
print(f"  recall:             {recall:.3f}   (caught {recall*100:.0f}% of all frauds)")
print(f"  base fraud rate:    {base_rate*100:.3f}%")
print(f"  LIFT:               {lift:.1f}×   (flagged group is {lift:.1f}× richer in fraud)")

**Fraud share per cluster** — does any K-Means cluster hide more fraud than average?

In [ ]:
#@title 6c. Fraud share inside each cluster { display-mode: "form" }
need("kmeans", "X_feat", "answer_key")
# assign every row (not just the sample) to its nearest cluster
clusters_all = kmeans.predict(X_feat)
df_cl = pd.DataFrame({"cluster": clusters_all, "fraud": answer_key})
# per-cluster: mean of 0/1 labels = fraud rate, count = cluster size
share = df_cl.groupby("cluster")["fraud"].agg(["mean", "size"])
share.columns = ["fraud_rate", "count"]
print("K-Means clusters — fraud share vs base rate "
      f"({answer_key.mean()*100:.3f}%):")
display(share.style.format({"fraud_rate": "{:.3%}"}))

**The lesson:** K-Means *groups* similar transactions; Isolation Forest finds *outliers*.
Neither knows what fraud is — the machine only finds "different". **Humans must review the
flags** — that's why precision matters: every false alarm costs analyst time.

## 7 · Tune the decision — the cost game

`flag_pct` is our operating threshold: missed fraud = **$500**, false alarm = **$5**.
Same trade-off as Lab 1 — but here the dial is "how weird is weird enough?"

**Sweep flag_pct from 0.5% to 5%** — total the cost for each and find the cheapest.

In [ ]:
#@title 7a. Sweep flag_pct — precision / recall / cost { display-mode: "form" }
need("scores", "answer_key")   # needs the Isolation Forest scores from §5
COST_FRAUD_MISSED, COST_FALSE_ALARM = 500, 5   # dollars per mistake type

total_fraud = int(answer_key.sum())   # all real frauds (473)
N = len(answer_key)
base_rate = total_fraud / N

# try every flag_pct the slider allows: 0.5, 1.0, ... 5.0
rows = []
for pct in np.arange(0.5, 5.5, 0.5):
    fl = scores <= np.quantile(scores, pct / 100)   # bottom pct% = flagged
    caught = int(answer_key[fl].sum())               # flagged rows that are fraud
    n_fl = int(fl.sum())
    missed = total_fraud - caught                    # frauds we let through
    false = n_fl - caught                            # legit flagged as fraud
    precision = caught / n_fl
    recall = caught / total_fraud
    lift = precision / base_rate
    cost = missed * COST_FRAUD_MISSED + false * COST_FALSE_ALARM
    rows.append({"flag_pct": pct, "flagged": n_fl, "caught": caught,
                 "precision": precision, "recall": recall, "lift": lift,
                 "missed": missed, "false_alarms": false, "cost_$": cost})

sweep = pd.DataFrame(rows)
best = sweep.loc[sweep["cost_$"].idxmin()]           # cheapest operating point

print(sweep.to_string(index=False,
      formatters={"precision": "{:.3f}".format, "recall": "{:.3f}".format,
                   "lift": "{:.1f}".format, "cost_$": "${:,.0f}".format}))
print(f"\nCheapest flag_pct by sweep: {best['flag_pct']:.1f}% → ${best['cost_$']:,.0f}")
print("Now drag the slider — does 1% win?")

**Pick a flag_pct and re-run** — does 1% win? (The sweep table above already shows every value.)

In [ ]:
#@title 7b. flag_pct — cost breakdown { display-mode: "form" }
#@param {"type":"slider","min":0.5,"max":5,"step":0.5,"label":"flag top X% most unusual"}
cost_pct = 1.0   # pick a value, re-run this cell to see its cost

need("scores", "answer_key")
COST_FRAUD_MISSED, COST_FALSE_ALARM = 500, 5   # dollars per mistake type

fl = scores <= np.quantile(scores, cost_pct / 100)   # flag bottom cost_pct%
caught = int(answer_key[fl].sum())
missed = int(answer_key.sum()) - caught
false = int(fl.sum()) - caught
print(f"flag_pct = {cost_pct:.1f}%")
print(f"  flagged:        {int(fl.sum()):,}")
print(f"  frauds caught:  {caught}  →  missed {missed}  →  ${missed*COST_FRAUD_MISSED:,}")
print(f"  false alarms:   {false}  →  ${false*COST_FALSE_ALARM:,}")
print(f"  TOTAL COST:     ${missed*COST_FRAUD_MISSED + false*COST_FALSE_ALARM:,}")
print("\nTip: the cheapest value is in the sweep table above (§7a).")

<details><summary>💡 Why isn't the cheapest flag_pct always the smallest?</summary>

Small flag_pct = high precision but you miss most frauds → $500 each adds up fast.
Large flag_pct = low precision → thousands of $5 false alarms. The sweep finds the
sweet spot where one more caught fraud is worth the extra alarms.</details>

## 8 · Export the model

Bundle the fitted models (and the scaler they were trained on) into one file in `models/` so you can reload them later.

In [ ]:
#@title Export the fitted models { display-mode: "form" }
import joblib, os
os.makedirs("models", exist_ok=True)

# pick up every model whose section was run (skips any you didn't reach)
models = {k: globals()[k] for k in ("kmeans", "pca", "iforest") if k in globals()}
if not models:
    raise SystemExit("No fitted models found — run §3–§5 first.")

# the scaler was applied inline in §2 and never stored as an object —
# refit the same scaler on the same cleaned data so the bundle is self-contained
scaler = None
if globals().get("use_scaling") and "X_all" in globals():
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler().fit(X_all)

bundle = {"use_scaling": bool(globals().get("use_scaling", False)),
          "scaler": scaler,
          "models": models}
path = "models/lab2_unsupervised.pkl"
joblib.dump(bundle, path)
print(f"saved {path}")
print("Contents:", {k: type(v).__name__ for k, v in bundle["models"].items()})
print("\nReload later with:")
print("  bundle = joblib.load('models/lab2_unsupervised.pkl')")
print("  # transform new data with bundle['scaler'] first if use_scaling is True")

## 9 · Takeaways

- No labels? **K-Means** finds groups, **Isolation Forest** finds outliers — both unsupervised.
- Neither knows what fraud *is*; they find "different". Humans review the flags.
- **Lift** = how much richer the flagged group is vs the base rate — the number that matters.
- Pick **flag_pct by cost**, not habit — same lesson as Lab 1's threshold.

🎉 Lab 2 complete.